 # AI Ruby on Rails Code Review

 This agent helps to review pull requests in minutes, it suggest best practices, coding convention, security vectors and linters. 

# Install dependencies

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


# Imports

In [ ]:
from src.github import download_pr
from src.parser import parse_diff
from src.rag import load_vector_db
from src.reviewer import review

/Users/renzodiaz/.local/share/mise/installs/python/3.14.6/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load DB

In [3]:
db = load_vector_db()

print(f"RAG database is ready.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 19148.68it/s]


RAG database is ready.


# Download Pull Request

In [4]:
PR_URL = "https://github.com/renzodiaz/notes-api/pull/4"

diff = download_pr(PR_URL)

print(f"Downloaded {len(diff)} characters.")

Downloaded 5625 characters.


# Parse the Pull Request

In [ ]:
parsed = parse_diff(diff)

print("Production Ruby files:")

for file in parsed["production_files"]:
    print("-", file["filename"])

print()
print("Test files:")

for file in parsed["test_files"]:
    print("-", file["filename"])

print(parsed["added_ruby_code"])

# Review & Print

In [ ]:
review_result = review(
    changed_code=parsed["added_ruby_code"],
    db=db
)

print(review_result)

# Overall Review

## Summary

This change adds a basic email/password login endpoint using `has_secure_password`, introduces a `User` model password digest column, and wires up a `/api/v1/login` route.

## Issues

### [High] Login action renders success and failure responses on successful authentication
Category: Rails / Maintainability

File:
`app/controllers/api/v1/auth_controller.rb`

Explanation:
The `login` action does not return after rendering the success response. As written, when the credentials are valid it will execute:

1. `render json: { user: user }, status: :ok`
2. then immediately continue to `render json: { error: "Invalid email or password" }, status: :unauthorized`

In Rails, this will typically raise a `DoubleRenderError` or produce broken behavior. This makes the login endpoint unusable for successful logins.

Recommendation:
Return immediately after the success render, or structure the action with an `else` branch.

Example:
```ruby
if user&.authenticate(params[:p